# OpenAI Embeddings — Humanized Learning Notebook

**Flow:** Environment setup → OpenAI embeddings → single text → multiple texts → similarity → semantic search.

The examples are written to show what each code block is doing rather than only showing the final output.

## 1. Load the API Key Safely

Create a `.env` file:

```text
OPENAI_API_KEY=your_key_here
```

Never commit secrets to Git.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. Add it to the environment or .env file."
    )

## 2. Initialize the OpenAI Embedding Model

The uploaded notebook uses `text-embedding-3-small`.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

embeddings

## 3. Embed One Text

`embed_query()` converts one string into one vector.

In [ ]:
single_text = (
    "LangChain and RAG are amazing frameworks and projects to work on"
)

single_embedding = embeddings.embed_query(single_text)

print(f"Input: {single_text}")
print(f"Vector dimensions: {len(single_embedding)}")
print(f"Sample values: {single_embedding[:5]}")

## 4. Embed Multiple Texts

`embed_documents()` turns each input string into its corresponding embedding.

In [ ]:
multiple_texts = [
    "Python is a programming language",
    "LangChain is a framework for LLM applications",
    "Embeddings convert text to numbers",
    "Vectors can be compared for similarity"
]

multiple_embeddings = embeddings.embed_documents(multiple_texts)

print(f"Number of texts: {len(multiple_texts)}")
print(f"Number of embeddings: {len(multiple_embeddings)}")
print(f"Dimensions of first embedding: {len(multiple_embeddings[0])}")

## 5. Cosine Similarity

In [ ]:
import numpy as np


def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=float)
    vec2 = np.asarray(vec2, dtype=float)

    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)

    if norm_a == 0 or norm_b == 0:
        raise ValueError("Cosine similarity is undefined for a zero vector.")

    return dot_product / (norm_a * norm_b)

## 6. Compare All Sentence Pairs

We use `j = i + 1` so we do not repeat the same unordered pair twice.

In [ ]:
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in Python",
    "Python is my favorite programming language"
]

sentence_embeddings = embeddings.embed_documents(sentences)

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = cosine_similarity(
            sentence_embeddings[i],
            sentence_embeddings[j]
        )

        print(f"'{sentences[i]}'")
        print(f"vs '{sentences[j]}'")
        print(f"Similarity: {similarity:.3f}")
        print("-" * 70)

## 7. Semantic Search

Semantic search retrieves the most similar documents for a query.

The learning version below calculates all document embeddings directly. A production RAG system normally precomputes and stores those vectors in a vector index.

In [ ]:
documents = [
    "LangChain is a framework for developing applications powered by language models",
    "Python is a high-level programming language",
    "Machine learning is a subset of artificial intelligence",
    "Embeddings convert text into numerical vectors",
    "The weather today is sunny and warm"
]

query = "What is LangChain?"

In [ ]:
def semantic_search(
    query,
    documents,
    embeddings_model,
    top_k=3
):
    """Return the top-k documents by cosine similarity."""

    query_embedding = embeddings_model.embed_query(query)

    document_embeddings = embeddings_model.embed_documents(
        documents
    )

    similarities = []

    for index, document_embedding in enumerate(
        document_embeddings
    ):
        score = cosine_similarity(
            query_embedding,
            document_embedding
        )

        similarities.append(
            (score, documents[index])
        )

    similarities.sort(
        key=lambda item: item[0],
        reverse=True
    )

    return similarities[:top_k]

## 8. Try a Second Query

In [ ]:
query = "What are embeddings?"

results = semantic_search(
    query,
    documents,
    embeddings,
    top_k=3
)

for score, document in results:
    print(f"Score: {score:.3f} | {document}")

In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
embeddings

In [ ]:
# Single text embeddings
single_text = "Langchain and Rag are amazing frameworks and projects to work on"
single_embeddings = embeddings.embed_query(single_text)
print(len(single_embeddings))
print(single_embeddings)

In [ ]:
print("📝 Single Text Embedding:")
print(f"Input: {single_text}")
print(f"Output: Vector of {len(single_embeddings)} dimensions")
print(f"Sample values: {single_embeddings[:5]}")

In [ ]:
# Example 2: Multiple texts at once
multiple_texts = [
    "Python is a programming language",
    "LangChain is a framework for LLM applications",
    "Embeddings convert text to numbers",
    "Vectors can be compared for similarity"
]

In [ ]:
multiple_embeddings = embeddings.embed_documents(multiple_texts)

In [ ]:
multiple_embeddings

In [ ]:
print("\n📚 Multiple Text Embeddings:")
print(f"Number of texts: {len(multiple_texts)}")
print(f"Number of embeddings: {len(multiple_embeddings)}")
print(f"Each embedding size: {len(multiple_embeddings[0])}")

In [ ]:
multiple_embeddings[0]

In [ ]:

from langchain_openai import OpenAIEmbeddings

# Different OpenAI embedding models
models_comparison = {
    "text-embedding-3-small": {
        "dimensions": 1536,
        "description": "Good balance of performance and cost",
        "cost_per_1m_tokens": 0.02,
        "use_case": "General purpose, cost-effective"
    },
    "text-embedding-3-large": {
        "dimensions": 3072,
        "description": "Highest quality embeddings",
        "cost_per_1m_tokens": 0.13,
        "use_case": "When accuracy is critical"
    },
    "text-embedding-ada-002": {
        "dimensions": 1536,
        "description": "Previous generation model",
        "cost_per_1m_tokens": 0.10,
        "use_case": "Legacy applications"
    }
}

# Display comparison
print("📊 OpenAI Embedding Models Comparison:\n")
for model_name, details in models_comparison.items():
    print(f"Model: {model_name}")
    print(f"  📏 Dimensions: {details['dimensions']}")
    print(f"  💰 Cost: ${details['cost_per_1m_tokens']}/1M tokens")
    print(f"  📝 Description: {details['description']}")
    print(f"  🎯 Use case: {details['use_case']}\n")

In [ ]:
# Example 1: Finding similar sentences
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in Python",
    "Python is my favorite programming language"
]

In [ ]:
import numpy as np


def cosine_similarity(vec1, vec2):
    """
    Cosine similarity measures the angle between two vectors.
    - Result close to 1: Very similar
    - Result close to 0: Not related
    - Result close to -1: Opposite meanings
    """

    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    return dot_product/(norm_a * norm_b)

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

In [ ]:
sentence_embeddings = embeddings.embed_documents(sentences)
sentence_embeddings

In [ ]:
# Calculate the simialrity betwween all pairs

for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        similarity = cosine_similarity(
            sentence_embeddings[i], sentence_embeddings[j])

        print(f"'{sentences[i]}' vs '{sentences[j]}'")
        print(f"Similarity: {similarity:.3f}\n")

In [ ]:
# Example- Semantic Search- Retireve the similar sentence
# Test semantic search
documents = [
    "LangChain is a framework for developing applications powered by language models",
    "Python is a high-level programming language",
    "Machine learning is a subset of artificial intelligence",
    "Embeddings convert text into numerical vectors",
    "The weather today is sunny and warm"
]
query = "What is Langchain?"

In [ ]:
def semantic_search(query, documents, embeddings_models, top_k=3):
    """Simple semantic search implementation"""

    # embed query and doument

    query_embedding = embeddings_models.embed_query(query)
    doc_embeddings = embeddings_models.embed_documents(documents)

    # Calculate the similarity score

    similarties = []

    for i, doc_emb in enumerate(doc_embeddings):
        similarity = cosine_similarity(query_embedding, doc_emb)
        similarties.append((similarity, documents[i]))

    # Sort by similarity
    similarties.sort(reverse=True)
    return similarties[:top_k]

In [ ]:
results = semantic_search(query, documents, embeddings)
results

In [ ]:
print(f"\n🔎 Semantic Search Results for: '{query}'")
for score, doc in results:
    print(f"Score: {score:.3f} | {doc}")

In [ ]:
query = "What is Embeddings?"
results = semantic_search(query, documents, embeddings)
results

## 9. OpenAI Model Comparison — Uploaded Notebook Snapshot

The uploaded notebook records:

- `text-embedding-3-small` → 1536 dimensions
- `text-embedding-3-large` → 3072 dimensions
- `text-embedding-ada-002` → 1536 dimensions

The original notebook also contains historical pricing values.

Treat these as course snapshots. Verify current model availability and pricing before production use.


## 10. Production Insight

Learning version:

```text
query
 ↓
re-embed every document
 ↓
compare
```

Production-style version:

```text
INGESTION
documents → embeddings → vector store

QUERY
query → embedding → vector search → top-k
```

That is the key scalability difference.


## 11. Final Takeaways

- Embeddings convert text into vectors.
- `embed_query()` is typically used for the query.
- `embed_documents()` converts the document/chunk collection.
- Cosine similarity compares vector alignment.
- Semantic search ranks candidate documents by similarity.
- Vector databases make this retrieval pattern scalable.